# 复制 `work_0213_eval_all_s2` 到 `eval_log`

这个 notebook 需要在 `conda` 的 `zkj-work` 环境里运行。

会执行两件事：
1. 把源项目里的 `model_1226` 复制到 `kejian-zhao-tsinghua-university/eval_log`。
2. 把 `s3_0227` 和 `s3_0301` 两个 run 合并后，写成目标项目里的一个新 run，名称仍为 `s3_0227`。

额外规则：所有 `summary/work_tpir_at_far_1e-*` 都会改名为 `summary/work_0213_tpir_at_far_1e-*`。

In [4]:
import math
import sys

import pandas as pd
import wandb
from tqdm.auto import tqdm

print(sys.executable)
print('wandb', wandb.__version__)

ENTITY = 'kejian-zhao-tsinghua-university'
SOURCE_PROJECT = 'work_0213_eval_all_s2'
TARGET_PROJECT = 'eval_log'

COPY_SPECS = [
    {
        'source_names': ['model_1226'],
        'target_name': 'model_1226',
    },
    {
        'source_names': ['s3_0227', 's3_0301'],
        'target_name': 's3_0227',
    },
]

SUMMARY_KEY_PREFIX = 'summary/work_tpir_at_far_'
SUMMARY_KEY_REPLACEMENT = 'summary/work_0213_tpir_at_far_'
DROP_HISTORY_PREFIXES = ('system/', '_')
STEP_METRIC_CANDIDATES = ('trainer/global_step', 'step', 'epoch')


def project_path(project):
    return f'{ENTITY}/{project}'


def rename_key(key):
    if isinstance(key, str) and key.startswith(SUMMARY_KEY_PREFIX):
        return key.replace(SUMMARY_KEY_PREFIX, SUMMARY_KEY_REPLACEMENT, 1)
    return key


def is_missing_value(value):
    if value is None:
        return True
    if isinstance(value, float):
        return math.isnan(value)
    try:
        missing = pd.isna(value)
    except (TypeError, ValueError):
        return False
    return bool(missing) if isinstance(missing, bool) else False


def clean_config(config):
    if not isinstance(config, dict):
        return {}
    return {k: v for k, v in config.items() if not str(k).startswith('_')}


def dedupe_keep_order(items):
    result = []
    seen = set()
    for item in items:
        marker = repr(item)
        if marker in seen:
            continue
        seen.add(marker)
        result.append(item)
    return result


def get_run_by_name(api, project, run_name):
    matches = [run for run in api.runs(project_path(project)) if run.name == run_name]
    if not matches:
        raise ValueError(f'在 {project_path(project)} 中找不到 run: {run_name}')
    if len(matches) > 1:
        raise ValueError(
            f'在 {project_path(project)} 中找到多个同名 run: {run_name}, ids={[run.id for run in matches]}'
        )
    return matches[0]


def target_run_exists(api, run_name):
    return any(run.name == run_name for run in api.runs(project_path(TARGET_PROJECT)))


def fetch_history_df(run):
    rows = list(run.scan_history())
    history_df = pd.DataFrame(rows)
    if history_df.empty:
        return history_df

    drop_cols = [
        col for col in history_df.columns
        if any(col.startswith(prefix) for prefix in DROP_HISTORY_PREFIXES)
    ]
    history_df = history_df.drop(columns=drop_cols, errors='ignore')

    rename_map = {
        col: rename_key(col)
        for col in history_df.columns
        if rename_key(col) != col
    }
    if rename_map:
        history_df = history_df.rename(columns=rename_map)
    return history_df


def fetch_summary_dict(run):
    summary = {}
    for key, value in dict(run.summary).items():
        if str(key).startswith('_'):
            continue
        summary[rename_key(key)] = value
    return summary


def merge_configs(runs):
    merged = {}
    for run in runs:
        merged.update(clean_config(run.config))
    return merged


def merge_tags(runs):
    tags = []
    for run in runs:
        tags.extend(list(run.tags or []))
    tags.append(f'copied-from:{SOURCE_PROJECT}')
    if len(runs) > 1:
        tags.append('merged')
    return dedupe_keep_order(tags)


def merged_group(runs):
    groups = dedupe_keep_order([run.group for run in runs if run.group])
    return groups[0] if len(groups) == 1 else None


def build_notes(runs, target_name):
    lines = [
        f'Copied to {ENTITY}/{TARGET_PROJECT} as {target_name}.',
        f'Source project: {ENTITY}/{SOURCE_PROJECT}.',
    ]
    for run in runs:
        lines.append(f'- {run.name} ({run.id})')
    return '\n'.join(lines)


def find_step_metric(history_df):
    for candidate in STEP_METRIC_CANDIDATES:
        if candidate in history_df.columns and history_df[candidate].notna().any():
            return candidate
    return None


def build_payload(api, spec):
    runs = [get_run_by_name(api, SOURCE_PROJECT, run_name) for run_name in spec['source_names']]
    history_frames = [fetch_history_df(run) for run in runs]
    history_df = pd.concat(history_frames, ignore_index=True) if history_frames else pd.DataFrame()
    renamed_summary_keys = sorted({
        rename_key(key)
        for run in runs
        for key in dict(run.summary).keys()
        if not str(key).startswith('_') and rename_key(key) != key
    })
    return {
        'runs': runs,
        'target_name': spec['target_name'],
        'history_df': history_df,
        'summary': fetch_summary_dict(runs[-1]),
        'config': merge_configs(runs),
        'tags': merge_tags(runs),
        'group': merged_group(runs),
        'notes': build_notes(runs, spec['target_name']),
        'renamed_summary_keys': renamed_summary_keys,
    }


def preview_payload(payload):
    print('=' * 80)
    print(f"target: {payload['target_name']}")
    print(f"source runs: {[run.name for run in payload['runs']]}")
    print(f"source ids: {[run.id for run in payload['runs']]}")
    print(f"history rows: {len(payload['history_df'])}")
    print(f"history columns: {list(payload['history_df'].columns)}")
    print(f"step metric: {find_step_metric(payload['history_df'])}")
    print(f"summary size: {len(payload['summary'])}")
    print(f"renamed summary keys: {payload['renamed_summary_keys']}")
    print(f"tags: {payload['tags']}")
    print(f"group: {payload['group']}")


def configure_default_step_metric(history_df):
    step_metric = find_step_metric(history_df)
    if step_metric:
        wandb.define_metric(step_metric)
        wandb.define_metric('*', step_metric=step_metric)
    return step_metric


def upload_history(history_df):
    if history_df.empty:
        return
    for _, row in tqdm(history_df.iterrows(), total=len(history_df), desc='uploading'):
        log_dict = {}
        for col, value in row.items():
            if not is_missing_value(value):
                log_dict[col] = value
        if log_dict:
            wandb.log(log_dict)


def create_target_run(api, payload):
    if target_run_exists(api, payload['target_name']):
        raise ValueError(
            f"目标项目 {project_path(TARGET_PROJECT)} 中已经存在 run: {payload['target_name']}"
        )

    init_kwargs = {
        'entity': ENTITY,
        'project': TARGET_PROJECT,
        'name': payload['target_name'],
        'config': payload['config'],
        'tags': payload['tags'],
        'notes': payload['notes'],
    }
    if payload['group']:
        init_kwargs['group'] = payload['group']

    new_run = wandb.init(**init_kwargs)
    print(f"created: {new_run.id} | {new_run.url}")

    step_metric = configure_default_step_metric(payload['history_df'])
    print(f'step metric = {step_metric}')

    upload_history(payload['history_df'])

    for key, value in payload['summary'].items():
        wandb.run.summary[key] = value

    wandb.finish()
    print(f"finished: {payload['target_name']}")


/root/anaconda3/envs/zkj-work/bin/python
wandb 0.25.1


In [2]:
# 先执行这个 cell 检查将要复制/合并的内容。
api = wandb.Api()
payloads = [build_payload(api, spec) for spec in COPY_SPECS]
for payload in payloads:
    preview_payload(payload)


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


target: model_1226
source runs: ['model_1226']
source ids: ['dw5opy9n']
history rows: 21
history columns: ['summary/work_0213_tpir_at_far_1e-06', 'n_images_seen', 'summary/ijbc_all_tpir_at_far_1e-08', 'summary/ijbc_001_tpir_at_far_1e-10', 'summary/ijbc_001_tpir_at_far_1e-08', 'summary/ijbc_001_tpir_at_far_1e-05', 'step', 'summary/ijbc_001_tpir_at_far_1e-09', 'summary/work_1201_tpir_at_far_1e-06', 'summary/ijbc_all_tpir_at_far_1e-07', 'trainer/epoch', 'summary/ijbc_all_tpir_at_far_5e-07', 'summary/ijbc_001_tpir_at_far_1e-06', 'summary/work_0213_tpir_at_far_1e-07', 'summary/work_0213_tpir_at_far_1e-10', 'summary/work_1201_tpir_at_far_1e-08', 'summary/work_1201_tpir_at_far_1e-10', 'epoch', 'summary/ijbc_001_tpir_at_far_5e-07', 'summary/ijbc_all_tpir_at_far_1e-06', 'summary/work_0213_tpir_at_far_1e-09', 'summary/work_1201_tpir_at_far_1e-07', 'summary/work_1201_tpir_at_far_1e-09', 'summary/ijbc_001_tpir_at_far_1e-07', 'trainer/global_step', 'summary/ijbc_all_tpir_at_far_1e-09', 'summary/wor

In [3]:
# 确认 preview 没问题后，再执行这个 cell。
api = wandb.Api()
payloads = [build_payload(api, spec) for spec in COPY_SPECS]
for payload in payloads:
    create_target_run(api, payload)


wandb: Currently logged in as: kejian-zhao (kejian-zhao-tsinghua-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


created: 5wljyc8f | https://wandb.ai/kejian-zhao-tsinghua-university/eval_log/runs/5wljyc8f
step metric = trainer/global_step


uploading:   0%|          | 0/21 [00:00<?, ?it/s]

epoch,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
n_images_seen,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
step,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-05,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-06,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-07,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-08,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-09,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-10,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_5e-07,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+19,...


finished: model_1226


created: amd2hdi8 | https://wandb.ai/kejian-zhao-tsinghua-university/eval_log/runs/amd2hdi8
step metric = trainer/global_step


uploading:   0%|          | 0/20 [00:00<?, ?it/s]

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
n_images_seen,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
step,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-05,▇█▇█▇▆▆▆▅▅▄▄▃▂▁▂▁▂▁▁
summary/ijbc_001_tpir_at_far_1e-06,██▇▇▇▄▅▁▅▆▅█▃▅▆▆▇▆▇▅
summary/ijbc_001_tpir_at_far_1e-07,▇▆▅▇█▄▅▁▆▇▅█▅▆▆▅█▇▇▆
summary/ijbc_001_tpir_at_far_1e-08,▃▄▁▆▃▅█▆▅▆▄▇▄▃▄█▃▆▆▃
summary/ijbc_001_tpir_at_far_1e-09,▃▄▁▆▃▅█▆▅▆▄▇▄▃▄█▃▆▆▃
summary/ijbc_001_tpir_at_far_1e-10,▃▄▁▆▃▅█▆▅▆▄▇▄▃▄█▃▆▆▃
summary/ijbc_001_tpir_at_far_5e-07,█▇▅▆▇▃▅▁▅▆▃█▄▄▅▅▇▆▆▅
+19,...


finished: s3_0227


# 复制 `work_1201_eval_all_s2` 到 `eval_log`

这个部分也是在 `conda` 的 `zkj-work` 环境里运行。

会把下面 5 个 run 逐个复制到 `kejian-zhao-tsinghua-university/eval_log`：
- `s3_12_13`
- `s3_12_21`
- `s3_12_26`
- `s3_12_29`
- `s3_01_04`

并把所有 `summary/work_tpir_at_far_*` 改名为 `summary/work_1201_tpir_at_far_*`。

In [5]:
import math
import sys

import pandas as pd
import wandb
from tqdm.auto import tqdm

print(sys.executable)
print('wandb', wandb.__version__)

ENTITY_1201 = 'kejian-zhao-tsinghua-university'
SOURCE_PROJECT_1201 = 'work_1201_eval_all_s2'
TARGET_PROJECT_1201 = 'eval_log'

COPY_SPECS_1201 = [
    {'source_names': ['s3_12_13'], 'target_name': 's3_12_13'},
    {'source_names': ['s3_12_21'], 'target_name': 's3_12_21'},
    {'source_names': ['s3_12_26'], 'target_name': 's3_12_26'},
    {'source_names': ['s3_12_29'], 'target_name': 's3_12_29'},
    {'source_names': ['s3_01_04'], 'target_name': 's3_01_04'},
]

SUMMARY_KEY_PREFIX_1201 = 'summary/work_tpir_at_far_'
SUMMARY_KEY_REPLACEMENT_1201 = 'summary/work_1201_tpir_at_far_'
DROP_HISTORY_PREFIXES_1201 = ('system/', '_')
STEP_METRIC_CANDIDATES_1201 = ('trainer/global_step', 'step', 'epoch')


def project_path_1201(project):
    return f'{ENTITY_1201}/{project}'


def rename_key_1201(key):
    if isinstance(key, str) and key.startswith(SUMMARY_KEY_PREFIX_1201):
        return key.replace(SUMMARY_KEY_PREFIX_1201, SUMMARY_KEY_REPLACEMENT_1201, 1)
    return key


def is_missing_value_1201(value):
    if value is None:
        return True
    if isinstance(value, float):
        return math.isnan(value)
    try:
        missing = pd.isna(value)
    except (TypeError, ValueError):
        return False
    return bool(missing) if isinstance(missing, bool) else False


def clean_config_1201(config):
    if not isinstance(config, dict):
        return {}
    return {k: v for k, v in config.items() if not str(k).startswith('_')}


def dedupe_keep_order_1201(items):
    result = []
    seen = set()
    for item in items:
        marker = repr(item)
        if marker in seen:
            continue
        seen.add(marker)
        result.append(item)
    return result


def get_run_by_name_1201(api, project, run_name):
    matches = [run for run in api.runs(project_path_1201(project)) if run.name == run_name]
    if not matches:
        raise ValueError(f'在 {project_path_1201(project)} 中找不到 run: {run_name}')
    if len(matches) > 1:
        raise ValueError(
            f'在 {project_path_1201(project)} 中找到多个同名 run: {run_name}, ids={[run.id for run in matches]}'
        )
    return matches[0]


def target_run_exists_1201(api, run_name):
    return any(run.name == run_name for run in api.runs(project_path_1201(TARGET_PROJECT_1201)))


def fetch_history_df_1201(run):
    rows = list(run.scan_history())
    history_df = pd.DataFrame(rows)
    if history_df.empty:
        return history_df

    drop_cols = [
        col for col in history_df.columns
        if any(col.startswith(prefix) for prefix in DROP_HISTORY_PREFIXES_1201)
    ]
    history_df = history_df.drop(columns=drop_cols, errors='ignore')

    rename_map = {
        col: rename_key_1201(col)
        for col in history_df.columns
        if rename_key_1201(col) != col
    }
    if rename_map:
        history_df = history_df.rename(columns=rename_map)
    return history_df


def fetch_summary_dict_1201(run):
    summary = {}
    for key, value in dict(run.summary).items():
        if str(key).startswith('_'):
            continue
        summary[rename_key_1201(key)] = value
    return summary


def merge_configs_1201(runs):
    merged = {}
    for run in runs:
        merged.update(clean_config_1201(run.config))
    return merged


def merge_tags_1201(runs):
    tags = []
    for run in runs:
        tags.extend(list(run.tags or []))
    tags.append(f'copied-from:{SOURCE_PROJECT_1201}')
    return dedupe_keep_order_1201(tags)


def merged_group_1201(runs):
    groups = dedupe_keep_order_1201([run.group for run in runs if run.group])
    return groups[0] if len(groups) == 1 else None


def build_notes_1201(runs, target_name):
    lines = [
        f'Copied to {ENTITY_1201}/{TARGET_PROJECT_1201} as {target_name}.',
        f'Source project: {ENTITY_1201}/{SOURCE_PROJECT_1201}.',
    ]
    for run in runs:
        lines.append(f'- {run.name} ({run.id})')
    return '\n'.join(lines)


def find_step_metric_1201(history_df):
    for candidate in STEP_METRIC_CANDIDATES_1201:
        if candidate in history_df.columns and history_df[candidate].notna().any():
            return candidate
    return None


def build_payload_1201(api, spec):
    runs = [get_run_by_name_1201(api, SOURCE_PROJECT_1201, run_name) for run_name in spec['source_names']]
    history_frames = [fetch_history_df_1201(run) for run in runs]
    history_df = pd.concat(history_frames, ignore_index=True) if history_frames else pd.DataFrame()
    renamed_summary_keys = sorted({
        rename_key_1201(key)
        for run in runs
        for key in dict(run.summary).keys()
        if not str(key).startswith('_') and rename_key_1201(key) != key
    })
    return {
        'runs': runs,
        'target_name': spec['target_name'],
        'history_df': history_df,
        'summary': fetch_summary_dict_1201(runs[-1]),
        'config': merge_configs_1201(runs),
        'tags': merge_tags_1201(runs),
        'group': merged_group_1201(runs),
        'notes': build_notes_1201(runs, spec['target_name']),
        'renamed_summary_keys': renamed_summary_keys,
    }


def preview_payload_1201(payload):
    print('=' * 80)
    print(f"target: {payload['target_name']}")
    print(f"source runs: {[run.name for run in payload['runs']]}")
    print(f"source ids: {[run.id for run in payload['runs']]}")
    print(f"history rows: {len(payload['history_df'])}")
    print(f"history columns: {list(payload['history_df'].columns)}")
    print(f"step metric: {find_step_metric_1201(payload['history_df'])}")
    print(f"summary size: {len(payload['summary'])}")
    print(f"renamed summary keys: {payload['renamed_summary_keys']}")
    print(f"tags: {payload['tags']}")
    print(f"group: {payload['group']}")


def configure_default_step_metric_1201(history_df):
    step_metric = find_step_metric_1201(history_df)
    if step_metric:
        wandb.define_metric(step_metric)
        wandb.define_metric('*', step_metric=step_metric)
    return step_metric


def upload_history_1201(history_df):
    if history_df.empty:
        return
    for _, row in tqdm(history_df.iterrows(), total=len(history_df), desc='uploading'):
        log_dict = {}
        for col, value in row.items():
            if not is_missing_value_1201(value):
                log_dict[col] = value
        if log_dict:
            wandb.log(log_dict)


def create_target_run_1201(api, payload):
    if target_run_exists_1201(api, payload['target_name']):
        raise ValueError(
            f"目标项目 {project_path_1201(TARGET_PROJECT_1201)} 中已经存在 run: {payload['target_name']}"
        )

    init_kwargs = {
        'entity': ENTITY_1201,
        'project': TARGET_PROJECT_1201,
        'name': payload['target_name'],
        'config': payload['config'],
        'tags': payload['tags'],
        'notes': payload['notes'],
    }
    if payload['group']:
        init_kwargs['group'] = payload['group']

    new_run = wandb.init(**init_kwargs)
    print(f"created: {new_run.id} | {new_run.url}")

    step_metric = configure_default_step_metric_1201(payload['history_df'])
    print(f'step metric = {step_metric}')

    upload_history_1201(payload['history_df'])

    for key, value in payload['summary'].items():
        wandb.run.summary[key] = value

    wandb.finish()
    print(f"finished: {payload['target_name']}")


/root/anaconda3/envs/zkj-work/bin/python
wandb 0.25.1


In [6]:
# 先执行这个 cell 检查 1201 这批 run 的复制内容。
api_1201 = wandb.Api()
payloads_1201 = [build_payload_1201(api_1201, spec) for spec in COPY_SPECS_1201]
for payload in payloads_1201:
    preview_payload_1201(payload)


target: s3_12_13
source runs: ['s3_12_13']
source ids: ['pk6ssw2y']
history rows: 78
history columns: ['step', 'summary/ijbc_001_tpir_at_far_1e-07', 'summary/ijbc_all_tpir_at_far_1e-07', 'summary/ijbc_001_tpir_at_far_5e-07', 'summary/ijbc_001_tpir_at_far_1e-08', 'summary/ijbc_all_tpir_at_far_1e-09', 'val/ijbc/001_tpir_at_far_1e-08', 'summary/work_1201_tpir_at_far_1e-09', 'val/ijbc/001_tpir_at_far_1e-10', 'val/ijbc/all_tpir_at_far_1e-09', 'summary/ijbc_001_tpir_at_far_1e-05', 'val/ijbc/001_tpir_at_far_1e-06', 'val/ijbc/all_tpir_at_far_5e-07', 'val/ijbc/all_tpir_at_far_1e-05', 'val/ijbc/all_tpir_at_far_1e-07', 'summary/work_1201_tpir_at_far_1e-07', 'summary/ijbc_all_tpir_at_far_1e-10', 'val/work/tpir_at_far_1e-10', 'val/work/tpir_at_far_1e-09', 'val/work/tpir_at_far_1e-08', 'summary/ijbc_all_tpir_at_far_1e-06', 'trainer/epoch', 'summary/work_1201_tpir_at_far_1e-08', 'val/work/tpir_at_far_1e-07', 'summary/ijbc_001_tpir_at_far_1e-06', 'val/ijbc/all_tpir_at_far_1e-08', 'summary/ijbc_001_tpi

In [7]:
# 确认 preview 没问题后，再执行这个 cell 真正写入 eval_log。
api_1201 = wandb.Api()
payloads_1201 = [build_payload_1201(api_1201, spec) for spec in COPY_SPECS_1201]
for payload in payloads_1201:
    create_target_run_1201(api_1201, payload)


created: w00uwrr5 | https://wandb.ai/kejian-zhao-tsinghua-university/eval_log/runs/w00uwrr5
step metric = trainer/global_step


uploading:   0%|          | 0/78 [00:00<?, ?it/s]

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇██
n_images_seen,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
step,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-05,████▇▆▅▄▃▂▂▁▂▂▁▂▁▁▁▁▁▁▁▁▂▁
summary/ijbc_001_tpir_at_far_1e-06,▄▂█▆▄▁▅▃▆▄▃▃▄▃▄▃▃▃▃▃▃▃▃▃▄▃
summary/ijbc_001_tpir_at_far_1e-07,▂▄▇▄▅▄█▅▃▆▃▃▄▂▂▂▂▁▁▁▂▃▂▂▃▂
summary/ijbc_001_tpir_at_far_1e-08,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-09,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-10,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_5e-07,▄▁█▇▁▃▄▂▆▇▅▆▆▄▇▅▅▅▅▄▅▅▅▅▆▄
+33,...


finished: s3_12_13


created: wd5v1ilg | https://wandb.ai/kejian-zhao-tsinghua-university/eval_log/runs/wd5v1ilg
step metric = trainer/global_step


...

finished: s3_12_29


created: ndvogudv | https://wandb.ai/kejian-zhao-tsinghua-university/eval_log/runs/ndvogudv
step metric = trainer/global_step


uploading:   0%|          | 0/48 [00:00<?, ?it/s]

epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
n_images_seen,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
step,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-05,▇█▅▇▇▅▆▃▂▃▃▂▃▁▃▁
summary/ijbc_001_tpir_at_far_1e-06,▇▇▅█▇▁▅▅▂▅▅▄▇▆▄▅
summary/ijbc_001_tpir_at_far_1e-07,▅▄▅▆▇▄▅▆▁▄▇▅█▇▆▅
summary/ijbc_001_tpir_at_far_1e-08,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-09,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-10,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_5e-07,▅▄▃█▅▁▅▅▁▃▅▃▇▇▄▅
+33,...


finished: s3_01_04


# 复制 `work_0213_eval_all_s2/s3_0315` 到 `eval_log`

这个部分在 `conda` 的 `zkj-work` 环境里运行。

会把 `kejian-zhao-tsinghua-university/work_0213_eval_all_s2` 里的 `s3_0315` 复制到 `kejian-zhao-tsinghua-university/eval_log`，并把所有 `summary/work_tpir_at_far_*` 改成 `summary/work_0213_tpir_at_far_*`。

In [1]:
import math
import sys

import pandas as pd
import wandb
from tqdm.auto import tqdm

print(sys.executable)
print('wandb', wandb.__version__)

ENTITY_0213_S315 = 'kejian-zhao-tsinghua-university'
SOURCE_PROJECT_0213_S315 = 'work_0213_eval_all_s2'
TARGET_PROJECT_0213_S315 = 'eval_log'

COPY_SPECS_0213_S315 = [
    {'source_names': ['s3_0315'], 'target_name': 's3_0315'},
]

SUMMARY_KEY_PREFIX_0213_S315 = 'summary/work_tpir_at_far_'
SUMMARY_KEY_REPLACEMENT_0213_S315 = 'summary/work_0213_tpir_at_far_'
DROP_HISTORY_PREFIXES_0213_S315 = ('system/', '_')
STEP_METRIC_CANDIDATES_0213_S315 = ('trainer/global_step', 'step', 'epoch')


def project_path_0213_s315(project):
    return f'{ENTITY_0213_S315}/{project}'


def rename_key_0213_s315(key):
    if isinstance(key, str) and key.startswith(SUMMARY_KEY_PREFIX_0213_S315):
        return key.replace(SUMMARY_KEY_PREFIX_0213_S315, SUMMARY_KEY_REPLACEMENT_0213_S315, 1)
    return key


def is_missing_value_0213_s315(value):
    if value is None:
        return True
    if isinstance(value, float):
        return math.isnan(value)
    try:
        missing = pd.isna(value)
    except (TypeError, ValueError):
        return False
    return bool(missing) if isinstance(missing, bool) else False


def clean_config_0213_s315(config):
    if not isinstance(config, dict):
        return {}
    return {k: v for k, v in config.items() if not str(k).startswith('_')}


def dedupe_keep_order_0213_s315(items):
    result = []
    seen = set()
    for item in items:
        marker = repr(item)
        if marker in seen:
            continue
        seen.add(marker)
        result.append(item)
    return result


def get_run_by_name_0213_s315(api, project, run_name):
    matches = [run for run in api.runs(project_path_0213_s315(project)) if run.name == run_name]
    if not matches:
        raise ValueError(f'在 {project_path_0213_s315(project)} 中找不到 run: {run_name}')
    if len(matches) > 1:
        raise ValueError(
            f'在 {project_path_0213_s315(project)} 中找到多个同名 run: {run_name}, ids={[run.id for run in matches]}'
        )
    return matches[0]


def target_run_exists_0213_s315(api, run_name):
    return any(run.name == run_name for run in api.runs(project_path_0213_s315(TARGET_PROJECT_0213_S315)))


def fetch_history_df_0213_s315(run):
    rows = list(run.scan_history())
    history_df = pd.DataFrame(rows)
    if history_df.empty:
        return history_df

    drop_cols = [
        col for col in history_df.columns
        if any(col.startswith(prefix) for prefix in DROP_HISTORY_PREFIXES_0213_S315)
    ]
    history_df = history_df.drop(columns=drop_cols, errors='ignore')

    rename_map = {
        col: rename_key_0213_s315(col)
        for col in history_df.columns
        if rename_key_0213_s315(col) != col
    }
    if rename_map:
        history_df = history_df.rename(columns=rename_map)
    return history_df


def fetch_summary_dict_0213_s315(run):
    summary = {}
    for key, value in dict(run.summary).items():
        if str(key).startswith('_'):
            continue
        summary[rename_key_0213_s315(key)] = value
    return summary


def merge_configs_0213_s315(runs):
    merged = {}
    for run in runs:
        merged.update(clean_config_0213_s315(run.config))
    return merged


def merge_tags_0213_s315(runs):
    tags = []
    for run in runs:
        tags.extend(list(run.tags or []))
    tags.append(f'copied-from:{SOURCE_PROJECT_0213_S315}')
    return dedupe_keep_order_0213_s315(tags)


def merged_group_0213_s315(runs):
    groups = dedupe_keep_order_0213_s315([run.group for run in runs if run.group])
    return groups[0] if len(groups) == 1 else None


def build_notes_0213_s315(runs, target_name):
    lines = [
        f'Copied to {ENTITY_0213_S315}/{TARGET_PROJECT_0213_S315} as {target_name}.',
        f'Source project: {ENTITY_0213_S315}/{SOURCE_PROJECT_0213_S315}.',
    ]
    for run in runs:
        lines.append(f'- {run.name} ({run.id})')
    return '\n'.join(lines)


def find_step_metric_0213_s315(history_df):
    for candidate in STEP_METRIC_CANDIDATES_0213_S315:
        if candidate in history_df.columns and history_df[candidate].notna().any():
            return candidate
    return None


def build_payload_0213_s315(api, spec):
    runs = [get_run_by_name_0213_s315(api, SOURCE_PROJECT_0213_S315, run_name) for run_name in spec['source_names']]
    history_frames = [fetch_history_df_0213_s315(run) for run in runs]
    history_df = pd.concat(history_frames, ignore_index=True) if history_frames else pd.DataFrame()
    renamed_summary_keys = sorted({
        rename_key_0213_s315(key)
        for run in runs
        for key in dict(run.summary).keys()
        if not str(key).startswith('_') and rename_key_0213_s315(key) != key
    })
    return {
        'runs': runs,
        'target_name': spec['target_name'],
        'history_df': history_df,
        'summary': fetch_summary_dict_0213_s315(runs[-1]),
        'config': merge_configs_0213_s315(runs),
        'tags': merge_tags_0213_s315(runs),
        'group': merged_group_0213_s315(runs),
        'notes': build_notes_0213_s315(runs, spec['target_name']),
        'renamed_summary_keys': renamed_summary_keys,
    }


def preview_payload_0213_s315(payload):
    print('=' * 80)
    print(f"target: {payload['target_name']}")
    print(f"source runs: {[run.name for run in payload['runs']]}")
    print(f"source ids: {[run.id for run in payload['runs']]}")
    print(f"history rows: {len(payload['history_df'])}")
    print(f"history columns: {list(payload['history_df'].columns)}")
    print(f"step metric: {find_step_metric_0213_s315(payload['history_df'])}")
    print(f"summary size: {len(payload['summary'])}")
    print(f"renamed summary keys: {payload['renamed_summary_keys']}")
    print(f"tags: {payload['tags']}")
    print(f"group: {payload['group']}")


def configure_default_step_metric_0213_s315(history_df):
    step_metric = find_step_metric_0213_s315(history_df)
    if step_metric:
        wandb.define_metric(step_metric)
        wandb.define_metric('*', step_metric=step_metric)
    return step_metric


def upload_history_0213_s315(history_df):
    if history_df.empty:
        return
    for _, row in tqdm(history_df.iterrows(), total=len(history_df), desc='uploading'):
        log_dict = {}
        for col, value in row.items():
            if not is_missing_value_0213_s315(value):
                log_dict[col] = value
        if log_dict:
            wandb.log(log_dict)


def create_target_run_0213_s315(api, payload):
    if target_run_exists_0213_s315(api, payload['target_name']):
        raise ValueError(
            f"目标项目 {project_path_0213_s315(TARGET_PROJECT_0213_S315)} 中已经存在 run: {payload['target_name']}"
        )

    init_kwargs = {
        'entity': ENTITY_0213_S315,
        'project': TARGET_PROJECT_0213_S315,
        'name': payload['target_name'],
        'config': payload['config'],
        'tags': payload['tags'],
        'notes': payload['notes'],
    }
    if payload['group']:
        init_kwargs['group'] = payload['group']

    new_run = wandb.init(**init_kwargs)
    print(f"created: {new_run.id} | {new_run.url}")

    step_metric = configure_default_step_metric_0213_s315(payload['history_df'])
    print(f'step metric = {step_metric}')

    upload_history_0213_s315(payload['history_df'])

    for key, value in payload['summary'].items():
        wandb.run.summary[key] = value

    wandb.finish()
    print(f"finished: {payload['target_name']}")


/root/anaconda3/envs/zkj-work/bin/python
wandb 0.25.1


In [2]:
# 先执行这个 cell 检查 s3_0315 的复制内容。
api_0213_s315 = wandb.Api()
payloads_0213_s315 = [build_payload_0213_s315(api_0213_s315, spec) for spec in COPY_SPECS_0213_S315]
for payload in payloads_0213_s315:
    preview_payload_0213_s315(payload)


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


target: s3_0315
source runs: ['s3_0315']
source ids: ['ft1auwzz']
history rows: 24
history columns: ['summary/work_0213_tpir_at_far_1e-09', 'summary/ijbc_001_tpir_at_far_1e-05', 'summary/ijbc_all_tpir_at_far_1e-10', 'summary/ijbc_all_tpir_at_far_1e-06', 'trainer/epoch', 'summary/ijbc_001_tpir_at_far_5e-07', 'summary/ijbc_all_tpir_at_far_1e-07', 'summary/work_0213_tpir_at_far_1e-07', 'summary/work_0213_tpir_at_far_1e-08', 'summary/ijbc_all_tpir_at_far_1e-09', 'n_images_seen', 'step', 'summary/work_1201_tpir_at_far_1e-10', 'summary/work_1201_tpir_at_far_1e-07', 'summary/ijbc_all_tpir_at_far_1e-08', 'summary/work_0213_tpir_at_far_1e-06', 'summary/ijbc_001_tpir_at_far_1e-10', 'summary/work_0213_tpir_at_far_1e-10', 'summary/ijbc_all_tpir_at_far_1e-05', 'summary/work_1201_tpir_at_far_1e-06', 'epoch', 'summary/ijbc_all_tpir_at_far_5e-07', 'summary/ijbc_001_tpir_at_far_1e-06', 'summary/work_1201_tpir_at_far_1e-09', 'summary/IJBC_TPR@FPR0.01', 'summary/ijbc_001_tpir_at_far_1e-07', 'summary/work

In [3]:
# 确认 preview 没问题后，再执行这个 cell 真正写入 eval_log。
api_0213_s315 = wandb.Api()
payloads_0213_s315 = [build_payload_0213_s315(api_0213_s315, spec) for spec in COPY_SPECS_0213_S315]
for payload in payloads_0213_s315:
    create_target_run_0213_s315(api_0213_s315, payload)


wandb: Currently logged in as: kejian-zhao (kejian-zhao-tsinghua-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


created: vj0j1ixn | https://wandb.ai/kejian-zhao-tsinghua-university/eval_log/runs/vj0j1ixn
step metric = trainer/global_step


uploading:   0%|          | 0/24 [00:00<?, ?it/s]

epoch,▁▁▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▆▆▇▇▇██
n_images_seen,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
step,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/IJBC_TPR@FPR0.01,█▇▆▆▅▅▅▅▄▄▄▃▃▃▂▂▂▂▂▁▂▁▁▁
summary/ijbc_001_tpir_at_far_1e-05,██▇▇▇▅▆▆▅▅▅▄▃▃▃▂▃▃▂▂▂▂▁▁
summary/ijbc_001_tpir_at_far_1e-06,▅▆▆█▄▃▅▂▆▂▅▄▂▂▃▅▄▄▃▂▂▁▃▃
summary/ijbc_001_tpir_at_far_1e-07,▇▇▇█▆▃▆▃▇▃▅▆▂▃▄▆▆▅▃▂▃▁▄▃
summary/ijbc_001_tpir_at_far_1e-08,▄▅▆▃▄▃▃▆▄▅▇▄▆█▆█▅▃▄▃▅▆▁▅
summary/ijbc_001_tpir_at_far_1e-09,▄▅▆▃▄▃▃▆▄▅▇▄▆█▆█▅▃▄▃▅▆▁▅
summary/ijbc_001_tpir_at_far_1e-10,▄▅▆▃▄▃▃▆▄▅▇▄▆█▆█▅▃▄▃▅▆▁▅
+20,...


finished: s3_0315
